<a href="https://colab.research.google.com/github/divijdawar/CUDA/blob/main/CUDA_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%%writefile tiledMatrixMultiply.cu

#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <iostream>
#include <chrono>

#define BLOCK_SIZE 16

__global__ void tiledMatrixMultiply(
    const float* A,
    const float* B,
    float* C,
    int M, int N, int K
) {
    __shared__ float As[BLOCK_SIZE][BLOCK_SIZE];
    __shared__ float Bs[BLOCK_SIZE][BLOCK_SIZE];

    int bx = blockIdx.x, by = blockIdx.y;
    int tx = threadIdx.x, ty = threadIdx.y;

    int row = by * BLOCK_SIZE + ty;
    int col = bx * BLOCK_SIZE + tx;

    float sum = 0.0f;

    for (int tile = 0; tile < (K + BLOCK_SIZE - 1) / BLOCK_SIZE; tile++) {
        if (row < M && (tile * BLOCK_SIZE + tx) < K)
            As[ty][tx] = A[row * K + tile * BLOCK_SIZE + tx];
        else
            As[ty][tx] = 0.0f;

        if (col < N && (tile * BLOCK_SIZE + ty) < K)
            Bs[tx][ty] = B[(tile * BLOCK_SIZE + ty) * N + col];
        else
            Bs[tx][ty] = 0.0f;

        __syncthreads();

        #pragma unroll 4
        for (int k = 0; k < BLOCK_SIZE; k++)
            sum += As[ty][k] * Bs[k][tx];

        __syncthreads();
    }

    if (row < M && col < N)
        C[row * N + col] = sum;
}

void launchMatrixMultiply(
    const float* d_A,
    const float* d_B,
    float* d_C,
    int M, int N, int K
) {
    dim3 blockDim(BLOCK_SIZE, BLOCK_SIZE);
    dim3 gridDim((N + BLOCK_SIZE - 1) / BLOCK_SIZE, (M + BLOCK_SIZE - 1) / BLOCK_SIZE);

    tiledMatrixMultiply<<<gridDim, blockDim>>>(d_A, d_B, d_C, M, N, K);
}

void randomMatrix(float* mat, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        mat[i] = static_cast<float>(rand()) / RAND_MAX;
    }
}

int main() {
    int M = 12258, N = 12258, K = 12258; // Matrix sizes
    size_t bytes_A = M * K * sizeof(float);
    size_t bytes_B = K * N * sizeof(float);
    size_t bytes_C = M * N * sizeof(float);

    float *h_A, *h_B, *h_C;
    float *d_A, *d_B, *d_C;

    // Allocate memory on host
    h_A = (float*)malloc(bytes_A);
    h_B = (float*)malloc(bytes_B);
    h_C = (float*)malloc(bytes_C);

    // Initialize matrices with random values
    randomMatrix(h_A, M, K);
    randomMatrix(h_B, K, N);

    // Allocate memory on device
    cudaMalloc(&d_A, bytes_A);
    cudaMalloc(&d_B, bytes_B);
    cudaMalloc(&d_C, bytes_C);

    // CUDA events for timing
    cudaEvent_t start, stop;
    float milliseconds = 0;

    // Start measuring host-to-device transfer time
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    cudaMemcpy(d_A, h_A, bytes_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes_B, cudaMemcpyHostToDevice);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    std::cout << "Host to Device Transfer Time: " << milliseconds << " ms\n";

    // Start measuring kernel execution time
    cudaEventRecord(start);

    launchMatrixMultiply(d_A, d_B, d_C, M, N, K);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    std::cout << "Kernel Execution Time: " << milliseconds << " ms\n";

    // Compute GFLOPS (FLOP count = 2 * M * N * K)
    double gflops = (2.0 * M * N * K) / (milliseconds * 1e6);
    std::cout << "Performance: " << gflops << " GFLOPS\n";

    // Start measuring device-to-host transfer time
    cudaEventRecord(start);

    cudaMemcpy(h_C, d_C, bytes_C, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    std::cout << "Device to Host Transfer Time: " << milliseconds << " ms\n";

    // Cleanup
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Overwriting tiledMatrixMultiply.cu


In [11]:
!nvcc tiledMatrixMultiply.cu -o tiledMatrixMultiply

In [12]:
!./tiledMatrixMultiply

Host to Device Transfer Time: 259.571 ms
Kernel Execution Time: 8.78787 ms
Performance: 419184 GFLOPS
Device to Host Transfer Time: 454.617 ms
